In [1]:
import pandas as pd

In [2]:
import pandas as pd

url = "https://raw.githubusercontent.com/karlaazuniga/etl-data-pipeline2516662022/refs/heads/main/data/raw/A_notas.csv"

df = pd.read_csv(url)

df.head()


,id_nota,id_estudiante,id_curso,nota,periodo
0,N5000,E1102,C204,7.9,I-2025
1,N5001,E1179,C205,8.1,I-2025
2,N5002,E1092,C202,8.6,I-2025
3,N5003,E1014,C211,8.0,I-2025
4,N5004,E1106,C208,7.4,II-2025


In [3]:
df.shape
df.columns
df.info()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 330 entries, 0 to 329
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   id_nota        330 non-null    object
 1   id_estudiante  325 non-null    object
 2   id_curso       328 non-null    object
 3   nota           326 non-null    object
 4   periodo        330 non-null    object
dtypes: object(5)
memory usage: 13.0+ KB


,0
id_nota,0
id_estudiante,5
id_curso,2
nota,4
periodo,0


In [4]:
notas = df.copy()

# limpiar nombres de columnas
notas.columns = notas.columns.str.strip().str.lower()

# limpiar espacios en texto
for col in notas.select_dtypes(include='object').columns:
    notas[col] = notas[col].astype(str).str.strip()

# convertir vacíos en null
notas = notas.replace(r'^\s*$', pd.NA, regex=True)

# eliminar duplicados
notas = notas.drop_duplicates()

In [5]:
notas['periodo'] = pd.to_datetime(
    notas['periodo'],
    errors='coerce'
)

/tmp/ipykernel_10455/2479018921.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  notas['periodo'] = pd.to_datetime(


In [6]:
notas['nota'] = pd.to_numeric(
    notas['nota'],
    errors='coerce'
)

In [7]:
validos = notas[
    notas['periodo'].notna() &
    notas['nota'].notna()
].copy()

rechazados = notas[
    notas['periodo'].isna() |
    notas['nota'].isna()
].copy()

In [8]:
def motivo(row):
    motivos = []

    if pd.isna(row['periodo']):
        motivos.append("periodo_invalido")

    if pd.isna(row['nota']):
        motivos.append("nota_invalido")

    return ",".join(motivos)

rechazados["motivo_rechazo"] = rechazados.apply(motivo, axis=1)

In [9]:
validos.to_csv("notas_curated.csv", index=False)
rechazados.to_csv("notas_rejects.csv", index=False)